# VaR of Equity Portfolio

**FINM 36700 · Homework 3 · Group 22**  
Lucas Albín Lazcano, Avi Rastogi, Mengting Wang, and Yihan Kong

*Acknowledgements: ChatGPT 5.6 was used to help explain the concepts, organize the submission, and assist with coding the solutions.*

## Data

This problem uses weekly return data from `spx_returns_weekly.xlsx`.

The four stocks selected for the analysis are AAPL, META, NVDA, and TSLA.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm

returns = pd.read_excel(
    "spx_returns_weekly.xlsx",
    sheet_name="spx returns",
    index_col="date",
    parse_dates=True,
)

tickers = ["AAPL", "META", "NVDA", "TSLA"]
stock_returns = returns[tickers].dropna()
stock_returns.head()

# 1 Diversification

## 1.1

Using the full sample, calculate for each series the (unconditional)

- volatility
- empirical VaR (.05)
- empirical CVaR (.05)

Recall that by **empirical** we refer to the direct quantile estimation.

In [ ]:
def empirical_risk(series):
    var_05 = series.quantile(0.05)
    return pd.Series({
        "Annualized volatility": series.std() * np.sqrt(52),
        "VaR (5%)": var_05,
        "CVaR (5%)": series[series <= var_05].mean(),
    })

individual_stats = stock_returns.apply(empirical_risk).T
individual_stats.style.format("{:.2%}")

_Answer:_ TSLA has the largest stand-alone risk by every measure. Its annualized volatility is about 57.97%, its 5% weekly VaR is -11.70%, and its weekly CVaR is -14.78%. AAPL is the least volatile of the four, with annualized volatility of about 27.76%. The VaR figures give the return cutoff for the worst 5% of weeks, while CVaR is more negative because it averages the returns that are already beyond that cutoff.

## 1.2

Form an equally-weighted portfolio of the investments.

Calculate the statistics of 1.1 for this portfolio, and compare the results to the individual return statistics. What do you find? What is driving this result?

In [ ]:
equal_weighted_return = stock_returns.mean(axis=1)
portfolio_stats = empirical_risk(equal_weighted_return).to_frame(
    "Equal-weighted portfolio"
).T
portfolio_stats.style.format("{:.2%}")

_Answer:_ The equally weighted portfolio has annualized volatility of about 31.20%, a 5% weekly VaR of -6.07%, and a weekly CVaR of -8.42%. Its risk is much lower than a simple average of the four stand-alone volatilities, and its tail losses are also smaller than those of META, NVDA, and TSLA individually. The reduction comes from diversification: the four stocks do not move by exactly the same amount every week, so positive and negative firm-specific movements partly offset one another inside the portfolio. Since all four are large technology-related stocks, their returns remain correlated, so diversification reduces rather than eliminates the risk.

## 1.3

Re-calculate 1.2, but this time drop your most volatile asset, and replace the portion it was getting with 0. (You could imagine we are replacing the most volatile asset with a negligibly small risk-free rate.)

In comparing the answer here to 1.2, how much risk is your most volatile asset adding to the portfolio? Is this in line with the amount of risk we measured in the stand-alone risk-assessment of 1.1?

In [ ]:
most_volatile = individual_stats["Annualized volatility"].idxmax()

# Keep the original 25% weights. The removed stock's 25% weight earns zero.
return_without_most_volatile = (
    stock_returns.drop(columns=most_volatile).sum(axis=1) / len(tickers)
)
reduced_stats = empirical_risk(return_without_most_volatile).to_frame(
    f"Portfolio with {most_volatile} replaced by zero"
).T

comparison_13 = pd.concat([portfolio_stats, reduced_stats])
comparison_13.style.format("{:.2%}")

_Answer:_ TSLA is the most volatile stock, so its 25% allocation is replaced by a zero return. Annualized portfolio volatility falls from 31.20% to 21.69%, a reduction of about 9.51 percentage points. Weekly VaR improves from -6.07% to -4.23%, and weekly CVaR improves from -8.42% to -6.00%. TSLA clearly adds meaningful portfolio risk, but its contribution is far smaller than its 57.97% stand-alone volatility. This is consistent with diversification because portfolio risk depends on TSLA's weight and its covariance with the other holdings, not only on TSLA's own volatility.

# 2. Dynamic Measures

## 2.1

Let's measure the **conditional** statistics of the equally-weighted portfolio of 1.2, as of the end of the sample.

### Volatility

For each security, calculate the rolling volatility series, $\sigma_t$, with a window of $m=26$.

The value at $\sigma_t$ denotes the estimate using data through time $t-1$, and thus predicting the volatility at time $t$.

### Mean

Suppose we can approximate that the weekly mean return is zero.

### VaR

Calculate the **normal VaR** and **normal CVaR** for $q=.05$ and $\tau=1$ as of the end of the sample. Use the approximation, $z_{.05}=-1.65$.

Report

- volatility (annualized)
- normal VaR (.05)
- normal CVaR (.05)

How do these compare to the answers in 1.2?

In [ ]:
all_returns = stock_returns.copy()
all_returns["Equal-weighted portfolio"] = equal_weighted_return

# shift(1) makes the value dated t depend only on observations through t-1.
rolling_vol = all_returns.rolling(26).std().shift(1)
latest_weekly_vol = rolling_vol.iloc[-1]

z_05 = -1.65
portfolio_weekly_vol = latest_weekly_vol["Equal-weighted portfolio"]
dynamic_stats = pd.DataFrame({
    "Annualized volatility": [portfolio_weekly_vol * np.sqrt(52)],
    "Normal VaR (5%)": [z_05 * portfolio_weekly_vol],
    "Normal CVaR (5%)": [
        -norm.pdf(z_05) / 0.05 * portfolio_weekly_vol
    ],
}, index=["Equal-weighted portfolio"])

print("Latest annualized 26-week volatility estimates:")
display((latest_weekly_vol * np.sqrt(52)).to_frame("Annualized volatility").style.format("{:.2%}"))
print("Conditional portfolio risk estimates:")
display(dynamic_stats.style.format("{:.2%}"))

_Answer:_ At the end of the sample, the portfolio's 26-week volatility estimate is 23.31% annualized. Under the normal approximation, the one-week 5% VaR is -5.33% and the CVaR is -6.61%. All three conditional risk estimates are below the full-sample empirical results from 1.2, where volatility was 31.20%, VaR was -6.07%, and CVaR was -8.42%. The difference occurs because the conditional estimates use only the most recent 26 weeks and therefore reflect the calmer risk environment near the end of the sample. The empirical measures in 1.2 use the entire history and include more severe observations. The normal model also summarizes the distribution using volatility instead of directly using the observed left tail.

## 2.2

Backtest the VaR using the **hit test**. Namely, check how many times the realized return at $t$ was smaller than the VaR return calculated using $\sigma_t$ (where again remember the notation uses $\sigma_t$ as a volatility based on data through $t-1$.)

Report the percentage of “hits” using both the

- expanding volatility
- rolling volatility

In [ ]:
# Both forecasts are shifted so each VaR uses only information available before t.
expanding_sigma = equal_weighted_return.expanding(26).std().shift(1)
rolling_sigma = equal_weighted_return.rolling(26).std().shift(1)

expanding_var = z_05 * expanding_sigma
rolling_var = z_05 * rolling_sigma

backtest = pd.concat(
    [equal_weighted_return, expanding_var, rolling_var],
    axis=1,
    keys=["Realized return", "Expanding VaR", "Rolling VaR"],
).dropna()

hit_rates = pd.Series({
    "Expanding volatility": (backtest["Realized return"] < backtest["Expanding VaR"]).mean(),
    "Rolling volatility": (backtest["Realized return"] < backtest["Rolling VaR"]).mean(),
}, name="Hit rate")

hit_rates.to_frame().style.format("{:.2%}")

_Answer:_ The hit rate is about 4.39% with expanding volatility and 4.02% with rolling volatility. Both are close to, but slightly below, the 5% rate implied by a 5% VaR. Therefore, both models are somewhat conservative over this sample because actual returns breach the estimated VaR less often than 5% of the time. The rolling estimate produces fewer hits because it reacts more quickly to recent volatility changes, while the expanding estimate changes slowly as new observations receive less weight in an increasingly long history.